# ⚖️ مُدوِّن — منظومة وكلاء ذكية لتوثيق الجلسات القضائية
### Agenticthon - Multi-Agent System for Judicial Session Documentation

---

| الوكيل | المهمة | التقنية |
|---|---|---|
| 🎙️ وكيل التفريغ | تحويل الصوت لنص + تحديد المتحدثين | Whisper + pyannote |
| ⚖️ وكيل التحليل القانوني | استخراج الادعاءات والدفوع | Claude API |
| 📝 وكيل التلخيص | إنشاء محضر الجلسة | Claude API |
| ✅ وكيل المراجعة | ضبط الجودة والاكتمال | Claude API |
| 💬 المساعد التفاعلي | سؤال وجواب عن الجلسة | Claude API |

## 1️⃣ تثبيت المكتبات

In [ ]:
!pip install -q openai-whisper pyannote.audio torch torchaudio anthropic gradio pydub python-dotenv

## 2️⃣ إعداد المفاتيح
⚠️ **عدّل المفاتيح قبل التشغيل**

In [ ]:
import os

# ========== عدّل هنا ==========
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # ← ضع مفتاح Anthropic هنا
os.environ["HF_TOKEN"] = "hf_..."               # ← ضع مفتاح Hugging Face هنا
# ==============================

ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]
HF_TOKEN = os.environ["HF_TOKEN"]

WHISPER_MODEL = "large-v3"
WHISPER_LANGUAGE = "ar"
CLAUDE_MODEL = "claude-sonnet-4-20250514"

DEFAULT_SPEAKERS = {
    "SPEAKER_00": "القاضي",
    "SPEAKER_01": "المدعي",
    "SPEAKER_02": "المدعى عليه",
    "SPEAKER_03": "محامي المدعي",
    "SPEAKER_04": "محامي المدعى عليه",
}

print("✅ تم إعداد المفاتيح")

## 3️⃣ بروتوكول A2A — التواصل بين الوكلاء

In [ ]:
import uuid
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Callable


@dataclass
class A2AMessage:
    """رسالة A2A — الوحدة الأساسية للتواصل بين الوكلاء"""
    sender: str
    receiver: str
    payload: dict[str, Any]
    msg_type: str = "data"
    msg_id: str = field(default_factory=lambda: uuid.uuid4().hex[:12])
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    parent_id: str | None = None

    def to_dict(self) -> dict:
        return {
            "msg_id": self.msg_id, "parent_id": self.parent_id,
            "sender": self.sender, "receiver": self.receiver,
            "msg_type": self.msg_type, "payload_keys": list(self.payload.keys()),
            "timestamp": self.timestamp,
        }

    def reply(self, payload: dict, sender: str) -> "A2AMessage":
        return A2AMessage(
            sender=sender, receiver=self.sender,
            payload=payload, msg_type="response", parent_id=self.msg_id,
        )


class A2AProtocol:
    """ناقل الرسائل بين الوكلاء — Message Bus"""

    def __init__(self):
        self._subscribers: dict[str, Callable] = {}
        self._log: list[dict] = []

    def register(self, agent_name: str, handler: Callable):
        self._subscribers[agent_name] = handler

    def send(self, message: A2AMessage) -> Any:
        self._log.append(message.to_dict())
        handler = self._subscribers.get(message.receiver)
        if handler is None:
            raise ValueError(f"Agent '{message.receiver}' is not registered.")
        result = handler(message)
        if isinstance(result, A2AMessage):
            self._log.append(result.to_dict())
        return result

    def get_log(self) -> list[dict]:
        return list(self._log)

    def summary(self) -> str:
        lines = []
        for m in self._log:
            lines.append(f"[{m['timestamp']}] {m['sender']} → {m['receiver']} ({m['msg_type']})")
        return "\n".join(lines)


print("✅ تم بناء بروتوكول A2A")

## 4️⃣ الوكيل الأساسي (Base Agent)

In [ ]:
from abc import ABC, abstractmethod


class BaseAgent(ABC):
    """الكلاس الأساسي — كل وكيل يسجل نفسه على ناقل A2A"""

    def __init__(self, name: str, protocol: A2AProtocol):
        self.name = name
        self.protocol = protocol
        self.protocol.register(self.name, self.handle_message)
        print(f"   🤖 تم تسجيل الوكيل: {self.name}")

    @abstractmethod
    def handle_message(self, message: A2AMessage):
        ...

    def send_to(self, receiver: str, payload: dict, msg_type: str = "data"):
        msg = A2AMessage(sender=self.name, receiver=receiver, payload=payload, msg_type=msg_type)
        return self.protocol.send(msg)


print("✅ تم بناء الوكيل الأساسي")

## 5️⃣ وكيل التفريغ الصوتي 🎙️
**Whisper large-v3 + pyannote Speaker Diarization**

In [ ]:
import whisper
import torch
from pyannote.audio import Pipeline as DiarizationPipeline
from pydub import AudioSegment
import tempfile


def convert_audio(file_path: str) -> str:
    """تحويل أي صيغة صوتية إلى WAV 16kHz mono"""
    audio = AudioSegment.from_file(file_path)
    audio = audio.set_frame_rate(16000).set_channels(1)
    wav_path = os.path.join(tempfile.gettempdir(), "session_audio.wav")
    audio.export(wav_path, format="wav")
    return wav_path


class TranscriptionAgent(BaseAgent):
    """وكيل التفريغ الصوتي — يحوّل الصوت لنص ويحدد المتحدثين"""

    def __init__(self, protocol: A2AProtocol):
        super().__init__("transcription_agent", protocol)
        self._whisper_model = None
        self._diarization = None

    def _load_whisper(self):
        if self._whisper_model is None:
            print("   ⏳ تحميل نموذج Whisper large-v3...")
            self._whisper_model = whisper.load_model(WHISPER_MODEL)
            print(f"   ✅ Whisper جاهز (GPU: {next(self._whisper_model.parameters()).is_cuda})")
        return self._whisper_model

    def _load_diarization(self):
        if self._diarization is None:
            print("   ⏳ تحميل نموذج تحديد المتحدثين...")
            self._diarization = DiarizationPipeline.from_pretrained(
                "pyannote/speaker-diarization-3.1",
                use_auth_token=HF_TOKEN,
            )
            if torch.cuda.is_available():
                self._diarization.to(torch.device("cuda"))
            print("   ✅ نموذج تحديد المتحدثين جاهز")
        return self._diarization

    def transcribe(self, audio_path: str, speaker_map: dict | None = None) -> dict:
        wav_path = convert_audio(audio_path)
        
        # الخطوة 1: التفريغ الصوتي
        print("   🎙️ جاري التفريغ الصوتي بـ Whisper...")
        model = self._load_whisper()
        result = model.transcribe(wav_path, language=WHISPER_LANGUAGE, task="transcribe", verbose=False)
        print(f"   ✅ تم التفريغ — {len(result['segments'])} مقطع")

        # الخطوة 2: تحديد المتحدثين
        print("   👥 جاري تحديد المتحدثين...")
        pipeline = self._load_diarization()
        diarization = pipeline(wav_path)
        print("   ✅ تم تحديد المتحدثين")

        # الخطوة 3: الدمج
        speaker_map = speaker_map or DEFAULT_SPEAKERS
        transcript = self._merge(result["segments"], diarization, speaker_map)

        return {
            "raw_text": result["text"],
            "segments": transcript,
            "full_transcript": self._format_transcript(transcript),
        }

    def _merge(self, segments, diarization, speaker_map):
        merged = []
        for seg in segments:
            mid = (seg["start"] + seg["end"]) / 2
            speaker_id = "UNKNOWN"
            for turn, _, speaker in diarization.itertracks(yield_label=True):
                if turn.start <= mid <= turn.end:
                    speaker_id = speaker
                    break
            merged.append({
                "start": round(seg["start"], 2), "end": round(seg["end"], 2),
                "speaker_id": speaker_id, "speaker": speaker_map.get(speaker_id, speaker_id),
                "text": seg["text"].strip(),
            })
        return merged

    @staticmethod
    def _format_transcript(segments):
        lines = []
        for s in segments:
            lines.append(f"[{s['start']:.1f}s - {s['end']:.1f}s] {s['speaker']}: {s['text']}")
        return "\n".join(lines)

    def handle_message(self, message: A2AMessage):
        audio_path = message.payload.get("audio_path")
        speaker_map = message.payload.get("speaker_map")
        result = self.transcribe(audio_path, speaker_map)
        return message.reply(payload=result, sender=self.name)


print("✅ وكيل التفريغ الصوتي جاهز")
print(f"   GPU متاح: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   عدد GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")

## 6️⃣ وكيل التحليل القانوني ⚖️

In [ ]:
import json
import anthropic

LEGAL_SYSTEM_PROMPT = """أنت محلل قانوني متخصص في النظام القضائي السعودي.
مهمتك: تحليل محضر جلسة قضائية واستخراج المعلومات التالية بدقة.

أجب بصيغة JSON فقط بالهيكل التالي:
{
    "case_type": "نوع القضية",
    "claims": ["قائمة الادعاءات"],
    "defenses": ["قائمة الدفوع"],
    "evidence": ["الأدلة المذكورة"],
    "legal_articles": ["المواد النظامية المُشار إليها"],
    "requests": {
        "plaintiff": ["طلبات المدعي"],
        "defendant": ["طلبات المدعى عليه"]
    },
    "key_facts": ["الوقائع الجوهرية"],
    "contradictions": ["أي تناقضات في الأقوال"],
    "timeline": ["تسلسل الأحداث المذكورة"]
}

تعامل مع النص بدقة. إذا لم تجد معلومة معينة اكتب مصفوفة فارغة [].
لا تخترع معلومات غير موجودة في النص."""


class LegalAnalysisAgent(BaseAgent):
    """وكيل التحليل القانوني — يستخرج البنية القانونية من المحضر"""

    def __init__(self, protocol: A2AProtocol):
        super().__init__("legal_analysis_agent", protocol)
        self.client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

    def analyze(self, transcript: str) -> dict:
        print("   ⚖️ جاري التحليل القانوني...")
        response = self.client.messages.create(
            model=CLAUDE_MODEL, max_tokens=4096,
            system=LEGAL_SYSTEM_PROMPT,
            messages=[{"role": "user", "content": f"حلل محضر الجلسة القضائية التالي:\n\n{transcript}"}],
        )
        raw = response.content[0].text
        try:
            if "```json" in raw:
                raw = raw.split("```json")[1].split("```")[0]
            elif "```" in raw:
                raw = raw.split("```")[1].split("```")[0]
            result = json.loads(raw.strip())
            print("   ✅ تم التحليل القانوني")
            return result
        except json.JSONDecodeError:
            print("   ⚠️ تعذر تحليل JSON — إرجاع النص الخام")
            return {"raw_analysis": raw, "parse_error": True}

    def handle_message(self, message: A2AMessage):
        transcript = message.payload.get("full_transcript", "")
        result = self.analyze(transcript)
        return message.reply(payload={"legal_analysis": result}, sender=self.name)


print("✅ وكيل التحليل القانوني جاهز")

## 7️⃣ وكيل التلخيص 📝

In [ ]:
SUMMARY_SYSTEM_PROMPT = """أنت كاتب ضبط ذكي متخصص في إعداد محاضر الجلسات القضائية.

بناءً على النص المُفرّغ والتحليل القانوني المُقدّم لك، أنشئ:

1. **محضر الجلسة الرسمي**: وثيقة مُهيكلة تتضمن:
   - رقم القضية والتاريخ (إن وُجد)
   - أطراف الدعوى
   - ملخص ما دار في الجلسة
   - الطلبات المقدمة
   - القرارات المتخذة

2. **ملخص تنفيذي**: فقرة واحدة مختصرة للقاضي.

3. **الإجراءات القادمة**: قائمة بالنقاط المعلّقة والخطوات التالية.

اكتب بلغة قانونية رسمية واضحة."""


class SummaryAgent(BaseAgent):
    """وكيل التلخيص — ينشئ محضر جلسة مُهيكل"""

    def __init__(self, protocol: A2AProtocol):
        super().__init__("summary_agent", protocol)
        self.client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

    def summarize(self, transcript: str, legal_analysis: dict) -> str:
        print("   📝 جاري إنشاء محضر الجلسة...")
        analysis_text = json.dumps(legal_analysis, ensure_ascii=False, indent=2)
        response = self.client.messages.create(
            model=CLAUDE_MODEL, max_tokens=4096,
            system=SUMMARY_SYSTEM_PROMPT,
            messages=[{
                "role": "user",
                "content": (
                    f"## النص المُفرّغ للجلسة:\n{transcript}\n\n"
                    f"## التحليل القانوني:\n{analysis_text}\n\n"
                    "أنشئ محضر الجلسة والملخص التنفيذي والإجراءات القادمة."
                ),
            }],
        )
        print("   ✅ تم إنشاء المحضر")
        return response.content[0].text

    def handle_message(self, message: A2AMessage):
        transcript = message.payload.get("full_transcript", "")
        legal_analysis = message.payload.get("legal_analysis", {})
        result = self.summarize(transcript, legal_analysis)
        return message.reply(payload={"summary": result}, sender=self.name)


print("✅ وكيل التلخيص جاهز")

## 8️⃣ وكيل المراجعة ✅

In [ ]:
QA_SYSTEM_PROMPT = """أنت مراجع جودة متخصص في محاضر الجلسات القضائية.

مهمتك مراجعة المحضر المُنتَج والتحقق من:
1. اكتمال المعلومات
2. دقة الإشارات النظامية
3. تناقضات الأقوال
4. معلومات مفقودة

أجب بصيغة JSON:
{
    "completeness_score": 0-100,
    "issues": [
        {"type": "missing_info | contradiction | inaccuracy | suggestion", "description": "...", "severity": "high | medium | low"}
    ],
    "verified_articles": ["المواد التي تم التحقق منها"],
    "overall_assessment": "تقييم عام مختصر"
}"""


class QAAgent(BaseAgent):
    """وكيل المراجعة — يراجع المحضر ويتحقق من جودته"""

    def __init__(self, protocol: A2AProtocol):
        super().__init__("qa_agent", protocol)
        self.client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

    def review(self, transcript: str, summary: str, legal_analysis: dict) -> dict:
        print("   ✅ جاري مراجعة الجودة...")
        analysis_text = json.dumps(legal_analysis, ensure_ascii=False, indent=2)
        response = self.client.messages.create(
            model=CLAUDE_MODEL, max_tokens=4096,
            system=QA_SYSTEM_PROMPT,
            messages=[{
                "role": "user",
                "content": (
                    f"## النص الأصلي:\n{transcript}\n\n"
                    f"## التحليل القانوني:\n{analysis_text}\n\n"
                    f"## المحضر المُنتَج:\n{summary}\n\n"
                    "راجع المحضر وقدّم تقرير الجودة."
                ),
            }],
        )
        raw = response.content[0].text
        try:
            if "```json" in raw:
                raw = raw.split("```json")[1].split("```")[0]
            elif "```" in raw:
                raw = raw.split("```")[1].split("```")[0]
            result = json.loads(raw.strip())
            print("   ✅ تم تقرير المراجعة")
            return result
        except json.JSONDecodeError:
            return {"raw_review": raw, "parse_error": True}

    def handle_message(self, message: A2AMessage):
        transcript = message.payload.get("full_transcript", "")
        summary = message.payload.get("summary", "")
        legal_analysis = message.payload.get("legal_analysis", {})
        result = self.review(transcript, summary, legal_analysis)
        return message.reply(payload={"qa_review": result}, sender=self.name)


print("✅ وكيل المراجعة جاهز")

## 9️⃣ المساعد التفاعلي 💬

In [ ]:
CHATBOT_SYSTEM_PROMPT = """أنت مساعد ذكي متخصص في الجلسات القضائية.
لديك إمكانية الوصول إلى النص الكامل للجلسة والتحليل القانوني والمحضر.
أجب بدقة بناءً على المعلومات المتوفرة فقط.
إذا لم تكن المعلومة موجودة، قل ذلك بوضوح.
كن مختصراً ودقيقاً بلغة قانونية مهنية."""


class ChatbotAgent(BaseAgent):
    """المساعد التفاعلي — سؤال وجواب عن الجلسة"""

    def __init__(self, protocol: A2AProtocol):
        super().__init__("chatbot_agent", protocol)
        self.client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        self.session_data = None
        self.conversation_history = []

    def set_session_data(self, transcript: str, legal_analysis: dict, summary: str):
        self.session_data = {
            "transcript": transcript,
            "legal_analysis": legal_analysis,
            "summary": summary,
        }
        self.conversation_history = []

    def ask(self, question: str) -> str:
        if not self.session_data:
            return "لم يتم تحميل بيانات الجلسة بعد."

        analysis_text = json.dumps(self.session_data["legal_analysis"], ensure_ascii=False, indent=2)
        context = (
            f"## النص المُفرّغ:\n{self.session_data['transcript']}\n\n"
            f"## التحليل القانوني:\n{analysis_text}\n\n"
            f"## محضر الجلسة:\n{self.session_data['summary']}"
        )

        self.conversation_history.append({"role": "user", "content": question})
        messages = [
            {"role": "user", "content": f"بيانات الجلسة:\n{context}"},
            {"role": "assistant", "content": "تم تحميل بيانات الجلسة. يمكنك طرح أسئلتك."},
            *self.conversation_history,
        ]

        response = self.client.messages.create(
            model=CLAUDE_MODEL, max_tokens=2048,
            system=CHATBOT_SYSTEM_PROMPT, messages=messages,
        )
        answer = response.content[0].text
        self.conversation_history.append({"role": "assistant", "content": answer})
        return answer

    def handle_message(self, message: A2AMessage):
        msg_type = message.payload.get("type", "ask")
        if msg_type == "load_session":
            self.set_session_data(
                transcript=message.payload.get("full_transcript", ""),
                legal_analysis=message.payload.get("legal_analysis", {}),
                summary=message.payload.get("summary", ""),
            )
            return message.reply(payload={"status": "session_loaded"}, sender=self.name)
        elif msg_type == "ask":
            question = message.payload.get("question", "")
            answer = self.ask(question)
            return message.reply(payload={"answer": answer}, sender=self.name)


print("✅ المساعد التفاعلي جاهز")

## 🔟 الـ Pipeline الرئيسي — ربط كل الوكلاء

In [ ]:
class MudawwinPipeline:
    """
    Pipeline مُدوِّن الكامل:
    Audio → Transcription → Legal Analysis → Summary → QA → Chatbot
    """

    def __init__(self):
        print("🔧 تهيئة منظومة مُدوِّن...")
        self.protocol = A2AProtocol()
        self.transcription = TranscriptionAgent(self.protocol)
        self.legal_analysis = LegalAnalysisAgent(self.protocol)
        self.summary = SummaryAgent(self.protocol)
        self.qa = QAAgent(self.protocol)
        self.chatbot = ChatbotAgent(self.protocol)
        print("✅ جميع الوكلاء جاهزون!\n")

    def run(self, audio_path: str, speaker_map: dict | None = None) -> dict:
        results = {}
        print("=" * 60)
        print("🚀 بدء تحليل الجلسة القضائية")
        print("=" * 60)

        # Step 1: Transcription
        print("\n📌 الخطوة 1/5: التفريغ الصوتي")
        msg1 = A2AMessage(sender="pipeline", receiver="transcription_agent",
                          payload={"audio_path": audio_path, "speaker_map": speaker_map})
        resp1 = self.protocol.send(msg1)
        transcript_data = resp1.payload
        results["transcription"] = transcript_data
        full_transcript = transcript_data["full_transcript"]

        # Step 2: Legal Analysis
        print("\n📌 الخطوة 2/5: التحليل القانوني")
        msg2 = A2AMessage(sender="pipeline", receiver="legal_analysis_agent",
                          payload={"full_transcript": full_transcript})
        resp2 = self.protocol.send(msg2)
        legal_analysis = resp2.payload["legal_analysis"]
        results["legal_analysis"] = legal_analysis

        # Step 3: Summary
        print("\n📌 الخطوة 3/5: إنشاء المحضر")
        msg3 = A2AMessage(sender="pipeline", receiver="summary_agent",
                          payload={"full_transcript": full_transcript, "legal_analysis": legal_analysis})
        resp3 = self.protocol.send(msg3)
        summary = resp3.payload["summary"]
        results["summary"] = summary

        # Step 4: QA
        print("\n📌 الخطوة 4/5: مراجعة الجودة")
        msg4 = A2AMessage(sender="pipeline", receiver="qa_agent",
                          payload={"full_transcript": full_transcript, "summary": summary, "legal_analysis": legal_analysis})
        resp4 = self.protocol.send(msg4)
        results["qa_review"] = resp4.payload["qa_review"]

        # Step 5: Load Chatbot
        print("\n📌 الخطوة 5/5: تجهيز المساعد التفاعلي")
        msg5 = A2AMessage(sender="pipeline", receiver="chatbot_agent",
                          payload={"type": "load_session", "full_transcript": full_transcript,
                                   "legal_analysis": legal_analysis, "summary": summary})
        self.protocol.send(msg5)

        results["a2a_log"] = self.protocol.get_log()

        print("\n" + "=" * 60)
        print("🎉 اكتمل التحليل بنجاح!")
        print("=" * 60)
        print(f"\n📊 سجل A2A — عدد الرسائل: {len(results['a2a_log'])}")
        print(self.protocol.summary())

        return results

    def ask_chatbot(self, question: str) -> str:
        msg = A2AMessage(sender="user", receiver="chatbot_agent",
                         payload={"type": "ask", "question": question})
        resp = self.protocol.send(msg)
        return resp.payload["answer"]


# تهيئة Pipeline
pipeline = MudawwinPipeline()

## 🎨 واجهة Gradio التفاعلية

In [ ]:
import gradio as gr

# حالة عامة لحفظ النتائج
session_results = {}


def process_audio(audio_file, spk0, spk1, spk2, spk3, spk4):
    """معالجة الملف الصوتي عبر كل الوكلاء"""
    global session_results, pipeline
    
    # إعادة تهيئة Pipeline لكل جلسة جديدة
    pipeline = MudawwinPipeline()

    speaker_map = {
        "SPEAKER_00": spk0, "SPEAKER_01": spk1, "SPEAKER_02": spk2,
        "SPEAKER_03": spk3, "SPEAKER_04": spk4,
    }

    results = pipeline.run(audio_file, speaker_map=speaker_map)
    session_results = results

    # تجهيز المخرجات
    transcript_text = results["transcription"]["full_transcript"]
    analysis = results["legal_analysis"]
    summary = results["summary"]
    qa = results["qa_review"]

    # تنسيق التحليل القانوني
    if not analysis.get("parse_error"):
        analysis_md = f"""### نوع القضية: {analysis.get('case_type', 'غير محدد')}

#### 🔴 الادعاءات:
{chr(10).join('- ' + c for c in analysis.get('claims', []))}

#### 🟢 الدفوع:
{chr(10).join('- ' + d for d in analysis.get('defenses', []))}

#### 📎 الأدلة:
{chr(10).join('- ' + e for e in analysis.get('evidence', []))}

#### 📖 المواد النظامية:
{chr(10).join('- ' + a for a in analysis.get('legal_articles', []))}

#### 📌 طلبات المدعي:
{chr(10).join('- ' + r for r in analysis.get('requests', {}).get('plaintiff', []))}

#### 📌 طلبات المدعى عليه:
{chr(10).join('- ' + r for r in analysis.get('requests', {}).get('defendant', []))}

#### ⚠️ تناقضات:
{chr(10).join('- ' + x for x in analysis.get('contradictions', [])) or 'لا توجد'}
"""
    else:
        analysis_md = str(analysis)

    # تنسيق تقرير الجودة
    if not qa.get("parse_error"):
        score = qa.get('completeness_score', 0)
        issues_text = ""
        for issue in qa.get('issues', []):
            icon = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(issue.get('severity', ''), '⚪')
            issues_text += f"\n{icon} **{issue.get('type', '')}**: {issue.get('description', '')}"

        qa_md = f"""### نسبة الاكتمال: {score}%

#### التقييم العام:
{qa.get('overall_assessment', '')}

#### الملاحظات:
{issues_text or 'لا توجد ملاحظات — المحضر مكتمل ودقيق ✅'}
"""
    else:
        qa_md = str(qa)

    # تنسيق سجل A2A
    a2a_log = results.get("a2a_log", [])
    a2a_text = "### سجل تواصل الوكلاء (A2A Protocol)\n\n"
    a2a_text += "| الوقت | المُرسل | ← | المُستقبل | النوع |\n"
    a2a_text += "|---|---|---|---|---|\n"
    for entry in a2a_log:
        a2a_text += f"| {entry['timestamp'][:19]} | {entry['sender']} | → | {entry['receiver']} | {entry['msg_type']} |\n"

    return transcript_text, analysis_md, summary, qa_md, a2a_text


def chat_with_bot(message, history):
    """محادثة مع المساعد التفاعلي"""
    if not session_results:
        return "⚠️ يرجى تحليل ملف صوتي أولاً من تبويب 'تحليل الجلسة'"
    answer = pipeline.ask_chatbot(message)
    return answer


# ---- بناء الواجهة ----
with gr.Blocks(
    title="مُدوِّن — توثيق الجلسات القضائية",
    theme=gr.themes.Base(primary_hue="cyan", neutral_hue="slate"),
    css=".gradio-container { direction: rtl; text-align: right; }",
) as demo:

    gr.Markdown("""
    # ⚖️ مُدوِّن
    ### منظومة وكلاء ذكية لتوثيق وتحليل الجلسات القضائية
    **المسار:** Multi-Agent Systems + Agent-to-Agent (A2A) | **المجال:** أتمتة الأعمال والمؤسسات

    ---
    🎙️ التفريغ ← ⚖️ التحليل القانوني ← 📝 التلخيص ← ✅ المراجعة ← 💬 المساعد التفاعلي
    """)

    with gr.Tabs():
        # ---- تبويب التحليل ----
        with gr.TabItem("🎙️ تحليل الجلسة"):
            with gr.Row():
                with gr.Column(scale=1):
                    audio_input = gr.Audio(label="📁 ارفع التسجيل الصوتي", type="filepath")
                    gr.Markdown("### تخصيص المتحدثين")
                    spk0 = gr.Textbox(label="المتحدث 1", value="القاضي")
                    spk1 = gr.Textbox(label="المتحدث 2", value="المدعي")
                    spk2 = gr.Textbox(label="المتحدث 3", value="المدعى عليه")
                    spk3 = gr.Textbox(label="المتحدث 4", value="محامي المدعي")
                    spk4 = gr.Textbox(label="المتحدث 5", value="محامي المدعى عليه")
                    analyze_btn = gr.Button("🚀 ابدأ التحليل", variant="primary", size="lg")

            with gr.Tabs():
                with gr.TabItem("🎙️ التفريغ الصوتي"):
                    transcript_output = gr.Textbox(label="النص المُفرّغ", lines=20, show_copy_button=True)

                with gr.TabItem("⚖️ التحليل القانوني"):
                    analysis_output = gr.Markdown(label="التحليل القانوني")

                with gr.TabItem("📝 محضر الجلسة"):
                    summary_output = gr.Markdown(label="محضر الجلسة")

                with gr.TabItem("✅ تقرير الجودة"):
                    qa_output = gr.Markdown(label="تقرير الجودة")

                with gr.TabItem("🔗 سجل A2A"):
                    a2a_output = gr.Markdown(label="سجل تواصل الوكلاء")

            analyze_btn.click(
                fn=process_audio,
                inputs=[audio_input, spk0, spk1, spk2, spk3, spk4],
                outputs=[transcript_output, analysis_output, summary_output, qa_output, a2a_output],
            )

        # ---- تبويب المحادثة ----
        with gr.TabItem("💬 المساعد التفاعلي"):
            gr.Markdown("### اسأل عن أي تفصيل في الجلسة")
            chatbot = gr.ChatInterface(
                fn=chat_with_bot,
                examples=[
                    "ما هي طلبات المدعي؟",
                    "ما موقف المدعى عليه؟",
                    "ما المواد النظامية المذكورة؟",
                    "لخّص لي الجلسة في 3 أسطر",
                    "هل يوجد تناقض في أقوال الأطراف؟",
                ],
                submit_btn="إرسال",
                retry_btn="إعادة",
                undo_btn="تراجع",
                clear_btn="مسح",
            )

print("✅ الواجهة جاهزة — شغّل الخلية التالية لبدء التطبيق")

## 🚀 تشغيل التطبيق
شغّل الخلية التالية وافتح الرابط الذي سيظهر

In [ ]:
demo.launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=True,  # ← ينشئ رابط عام للمشاركة
    show_error=True,
)